# **DATA PIPELINE PORTFOLIO PROJECT**

In [2]:
import pandas as pd
import sqlite3
import logging
import sys
import csv
from datetime import datetime
import unittest

# *SETUP, Creating Loggers*

In [4]:
#This is used to clear existing handlers since in jupyter it keeps on creating it anytime we run the cells
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logger = logging.getLogger("main")
logger.setLevel(logging.DEBUG)

streamhandler =logging.StreamHandler(sys.stdout)
logger.addHandler(streamhandler)

filehandler =logging.FileHandler('newlogs.log')
logger.addHandler(filehandler)

Testchange_logger =logging.getLogger("testchange")
Testchange_logger.setLevel(logging.ERROR)
filehandler_2 = logging.FileHandler('change.log')
Testchange_logger.addHandler(filehandler_2)

#Formatting logs
formatter =logging.Formatter("[%(asctime)s] %(name)s %(message)s")
streamhandler.setFormatter(formatter)
filehandler.setFormatter(formatter)
filehandler_2.setFormatter(formatter)

# *!DATA INGESTION*

In [6]:
import os
print(os.getcwd())
print(os.path.exists(r"C:\Users\gelsc\jupyter Repository\subscriber-pipeline-starter-kit\subscriber-pipeline-starter-kit\dev\cademycode.db"))

C:\Users\gelsc\jupyter Repository\subscriber-pipeline-starter-kit\subscriber-pipeline-starter-kit\dev
True


In [7]:
logger.info ('Connecting to the cademycode.db for Data Ingestion')
db_path ="C:/Users/gelsc/jupyter Repository/subscriber-pipeline-starter-kit/subscriber-pipeline-starter-kit/dev/cademycode.db"
connection = sqlite3.connect(db_path)
cursor= connection.cursor()

[2025-09-14 12:16:43,552] main Connecting to the cademycode.db for Data Ingestion


**Read the tables into a pandas dataFrame**


In [9]:
cursor.execute('''SELECT name FROM sqlite_master
                  WHERE type = "table"; ''').fetchall()

[('cademycode_students',),
 ('cademycode_courses',),
 ('cademycode_student_jobs',)]

In [10]:
logger.info('Loading cademycode_students into a dataframe students_df')
try:
   students_df = pd.read_sql_query('''SELECT * FROM cademycode_students ''', connection)
except Exception:
    logger.error('Injecting from non-existent table')
students_df

[2025-09-14 12:16:43,582] main Loading cademycode_students into a dataframe students_df


,uuid,name,dob,sex,contact_info,job_id,num_course_taken,current_career_path_id,time_spent_hrs
0,1,Annabelle Avery,1943-07-03,F,"{""mailing_address"": ""303 N Timber Key, Irondal...",7.0,6.0,1.0,4.99
1,2,Micah Rubio,1991-02-07,M,"{""mailing_address"": ""767 Crescent Fair, Shoals...",7.0,5.0,8.0,4.4
2,3,Hosea Dale,1989-12-07,M,"{""mailing_address"": ""P.O. Box 41269, St. Bonav...",7.0,8.0,8.0,6.74
3,4,Mariann Kirk,1988-07-31,F,"{""mailing_address"": ""517 SE Wintergreen Isle, ...",6.0,7.0,9.0,12.31
4,5,Lucio Alexander,1963-08-31,M,"{""mailing_address"": ""18 Cinder Cliff, Doyles b...",7.0,14.0,3.0,5.64
...,...,...,...,...,...,...,...,...,...
4995,4996,Quentin van Harn,1967-07-07,N,"{""mailing_address"": ""591 Blue Berry, Coulee, I...",5.0,5.0,2.0,13.82
4996,4997,Alejandro van der Sluijs,1964-11-03,M,"{""mailing_address"": ""30 Iron Divide, Pewaukee ...",4.0,13.0,1.0,7.86
4997,4998,Brock Mckenzie,2004-11-25,M,"{""mailing_address"": ""684 Rustic Rest Avenue, C...",8.0,10.0,3.0,12.1
4998,4999,Donnetta Dillard,1943-02-12,N,"{""mailing_address"": ""900 Indian Oval, Euclid, ...",3.0,6.0,5.0,14.86


In [11]:
logger.info('Loading cademycode_courses into a dataframe courses_df')
try:
   courses_df = pd.read_sql_query('''SELECT * FROM cademycode_courses ''', connection)
except :
    raise logger.error('Injecting from non-existent table')
courses_df

[2025-09-14 12:16:43,653] main Loading cademycode_courses into a dataframe courses_df


,career_path_id,career_path_name,hours_to_complete
0,1,data scientist,20
1,2,data engineer,20
2,3,data analyst,12
3,4,software engineering,25
4,5,backend engineer,18
5,6,frontend engineer,20
6,7,iOS developer,27
7,8,android developer,27
8,9,machine learning engineer,35
9,10,ux/ui designer,15


In [12]:
courses_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   career_path_id     10 non-null     int64 
 1   career_path_name   10 non-null     object
 2   hours_to_complete  10 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 372.0+ bytes


In [13]:
logger.info('Loading cademycode_student_jobs into a dataframe jobs_df')
try:
   jobs_df =pd.read_sql_query(''' SELECT * FROM cademycode_student_jobs''', connection)
except:
    raise logger.error('injecting from non-existent table')
jobs_df

[2025-09-14 12:16:43,680] main Loading cademycode_student_jobs into a dataframe jobs_df


,job_id,job_category,avg_salary
0,1,analytics,86000
1,2,engineer,101000
2,3,software developer,110000
3,4,creative,66000
4,5,financial services,135000
5,6,education,61000
6,7,HR,80000
7,8,student,10000
8,9,healthcare,120000
9,0,other,80000


In [14]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   job_id        13 non-null     int64 
 1   job_category  13 non-null     object
 2   avg_salary    13 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 444.0+ bytes


In [15]:
logger.info('Clossing the Database after injecting data from the 3 tables')
connection.close()

[2025-09-14 12:16:43,703] main Clossing the Database after injecting data from the 3 tables


# *!DATA CLEANING*

In [17]:
students_df.columns

Index(['uuid', 'name', 'dob', 'sex', 'contact_info', 'job_id',
       'num_course_taken', 'current_career_path_id', 'time_spent_hrs'],
      dtype='object')

In [18]:
def change_D_type(df):
    try:
       df['num_course_taken'] =pd.to_numeric(df['num_course_taken'])
       df['time_spent_hrs'] =pd.to_numeric(df['time_spent_hrs'])
       df['job_id'] = pd.to_numeric(df['job_id'])
       df['current_career_path_id'] = pd.to_numeric(df['current_career_path_id'])
       if 'dob' in df.columns:
            df =df.rename(columns={'dob' : 'Date_of_Birth'})
    except KeyError as e :
        logger.error(f" some columns are missing {e}")
        raise ValueError(f"Required column '{e}' not found in DataFrame")
    finally:
        logger.info('Data types changed sucessful')
    return df


In [19]:
def remove_special_character(df):
    try:
        has_special =(df['contact_info'].str.contains('"mailing_address": "',case=False, na =False) | df['contact_info'].str.contains('[{|"]+', regex=True, na=False))
        if has_special.any():
           df['contact_info'] = df['contact_info'].replace('"mailing_address": "' , '', regex=True)
           df['contact_info'] = df['contact_info'].replace('[{}|"]+', '', regex=True)
           df['contact_info'] = df['contact_info'].str.replace(r'[,.]+\s*$', '', regex=True)
           df['email'] = df['contact_info'].str.extract(r'email:\s*([^\s,()]+)', expand=False)
           df=df.drop(columns=['contact_info'])
    except KeyError as e:
        logger.eror(f"column missing: {e}")
        raise ValueError(f"Required column '{e}' not found in DataFrame")
    except Excepetion as e:
        logger.error(f"Contact info does not conform to the normal uneditted formart '{e}' ")
        raise
    finally:
            logger.info('contact info processing completed')
    return df

In [20]:
def final_cleaning(df):
    try:
        logger.info('Starting Final Cleaning')
        
        df = change_D_type(df)
        logger.info('Columns Data type changed')

        df = remove_special_character(df)
        logger.info('special characters removed from contact info')

        df.fillna(0)
        logger.info('Missing values filled with 0')

        return df
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        raise
        

In [21]:
logger.debug("Executing final Cleaning function on dataFrame")
Cleaned_students_df = final_cleaning(students_df)

[2025-09-14 12:16:43,760] main Executing final Cleaning function on dataFrame
[2025-09-14 12:16:43,761] main Starting Final Cleaning
[2025-09-14 12:16:43,772] main Data types changed sucessful
[2025-09-14 12:16:43,773] main Columns Data type changed
[2025-09-14 12:16:43,818] main contact info processing completed
[2025-09-14 12:16:43,819] main special characters removed from contact info
[2025-09-14 12:16:43,822] main Missing values filled with 0


In [22]:
Cleaned_courses_df = courses_df

In [23]:
Cleaned_jobs_df = jobs_df

# *Unittesting*

In [25]:
class TextCleanedData(unittest.TestCase):

    def Test_Course_Schema(self):
        self.assertEqual(list(self.courses_df.columns), list(self.Cleaned_courses_df.columns), "Expected the columns of courses_df.columns and Cleaned_courses_df to be the same")
        
    def Test_jobs_Schema(self):
        self.assertEqual(list(self.jobs_df.columns), list(self.Cleaned_jobs_df.columns), 'Expected the columns of jobs_df and Cleaned_jobs_df to be the same')

    def Test_Student_cols(self):
        self.assertIn("uuid", self.Cleaned_students_df.columns, 'Expected uuid columns to be available in cleaned_student_df')
        self.asserIn("name", self.Cleaned_students_df.columns, 'Expected name columns to be available in cleaned_student_df')
        
    def Text_new_data_exist(self):
        assertLess(len(self.Cleaned_students_df), len(students_df), 'New Data is added to the student_df')
        assertLess(len(self.Cleaned_courses_df), len(self.courses_df), 'New Data is added to the courses_df')
        assertLess(len(self.Cleaned_jobs_df), len(self.jobs_df), 'New Data is added to the courses_df')

unittest.main(argv=[''], exit=False)
#This was created to log only AssertionErrors and failure from unittest to the change.log file
runner = unittest.TextTestRunner(stream=sys.stdout, verbosity=2)
result = runner.run(unittest.defaultTestLoader.loadTestsFromTestCase(TextCleanedData))
for test, err in result.failures + result.errors:
    Testchange_logger.error("Test %s failed: \n%s", test, err)


----------------------------------------------------------------------
Ran 0 tests in 0.000s

NO TESTS RAN



----------------------------------------------------------------------
Ran 0 tests in 0.000s

NO TESTS RAN


# *DATA UPDATING*

In [27]:
logger.info ('Connecting to cleanedData.db')
cleanedDB_path = "C:/Users/gelsc/jupyter Repository/subscriber-pipeline-starter-kit/subscriber-pipeline-starter-kit/dev/cleanedData_.db"
conn = sqlite3.connect(cleanedDB_path)
cursor = conn.cursor()

[2025-09-14 12:16:43,896] main Connecting to cleanedData.db


In [28]:
cursor.execute('''SELECT name FROM sqlite_master
                WHERE type = 'table' ''').fetchall()

[('cademycode_students',),
 ('cademycode_courses',),
 ('cademycode_student_jobs',)]

In [29]:
cursor.execute('''CREATE TABLE IF NOT EXISTS cademycode_students(
               uuid INTEGER,
               name TEXT,
               Date_of_Birth TEXT,
               sex TEXT,
               jobs_id REAL
               num_course_taken REAL,
               current_career_path_id REAL,
               time_spent_hrs REAL,
               email TEXT
             )
                ''')
              

In [30]:
logger.info('Replaces the data in cademycode_students table with cleaned students_df')
if len(Cleaned_students_df) == len(students_df):
    Cleaned_students_df.to_sql('cademycode_students', conn, if_exists ='replace', index = False)
    logger.info('Successful Update')
elif len(Cleaned_students_df) < len(students_df):
    new_uuid = set(students_df['uuid']) - set(Cleaned_students_df['uuid'])
    for i in new_uuid:
        new_row = final_cleaning(students_df[students_df['uuid'] == i])
        Cleaned_students_df.pd.concat(new_row)  
    Cleaned_students_df.to_sql('cademycode_students', conn, if_exists ='replace', index = False)
    logger.info('Successful Update')
else:
    raise logger.error('Update Failed for cademycode_students table')
    raise Testchange_logger('Update Failed for cademycode_students table')

[2025-09-14 12:16:43,925] main Replaces the data in cademycode_students table with cleaned students_df
[2025-09-14 12:16:43,966] main Successful Update


In [31]:
logger.info('Creates cademycode_courses table in cleanedData.db')
   
cursor.execute(''' CREATE TABLE IF NOT EXISTS cademycode_courses (
                   career_path_id INTEGER,
                   career_path_name TEXT,
                   hours_to_complete INTEGER) '''
               )         

[2025-09-14 12:16:43,972] main Creates cademycode_courses table in cleanedData.db


In [32]:
logger.info('Replaces the data in cademycode_courses table with Cleaned_courses_df')
if len(Cleaned_courses_df) == len(courses_df):
    Cleaned_courses_df.to_sql('cademycode_courses',conn, if_exists = 'replace', index =False)
    logger.info('Successful Update')
elif len(Cleaned_courses_df) < len(courses_df):
    new_career_path_id = set(Cleaned_courses_df['career_path_id']) - set(courses_df['career_path_id'])
    for i in new_career_path_id:
        new_courses_row = courses_df[courses_df['career_path_id'] == i]
        Cleaned_courses_df.pd.concat(new_courses_row)
    Cleaned_courses_df.to_sql('cademycode_courses',conn, if_exists = 'replace', index =False)
    logger.info('Successful Update')
else:
    raise logger.error('Update Failed for cademycode_courses table')
    raise Testchange_logger.error('Update Failed for cademycode_courses table')
    

[2025-09-14 12:16:43,982] main Replaces the data in cademycode_courses table with Cleaned_courses_df
[2025-09-14 12:16:43,997] main Successful Update


In [33]:
logger.info('Creates cademycode_student_jobs table in cleanedData.db')
cursor.execute('''CREATE TABLE IF NOT EXISTS cademycode_student_jobs(
                job_id INTEGER,
                job_category TEXT,
                avg_salary REAL)'''
              )

[2025-09-14 12:16:44,004] main Creates cademycode_student_jobs table in cleanedData.db


In [34]:
logger.info('Replaces the data in  cademycode_student_jobs table with Cleaned_jobs_df')
if len(Cleaned_jobs_df) == len(jobs_df):
    Cleaned_jobs_df.to_sql('cademycode_student_jobs', conn, if_exists= 'replace', index=False)
    logger.info('Successful Update')
elif len(Cleaned_jobs_df) < len(jobs_df):
    new_job_id = set(Cleaned_jobs_df['job_id']) - set(jobs_df['job_id'])
    for i in new_job_id:
        new_job_row = jobs_df[jobs_df['jobs_id']== i ]
        Cleaned_jobs_df.pd.concat(new_job_row)
    Cleaned_jobs_df.to_sql('cademycode_student_jobs', conn, if_exists= 'replace', index=False)
    logger.info('Successful Update')
else:
    raise logger.error('Update Failed for cademycode_student_jobs table')
    raise Testchange_logger.error('Update Failed for cademycode_student_jobs table')

[2025-09-14 12:16:44,014] main Replaces the data in  cademycode_student_jobs table with Cleaned_jobs_df
[2025-09-14 12:16:44,031] main Successful Update


In [35]:
Analytics_table_df =pd.read_sql_query(''' SELECT uuid, name, Date_of_Birth, sex, email, num_course_taken, career_path_name, hours_to_complete, 
                                          job_category, avg_salary
                                          FROM cademycode_students AS cs
                                         
                                         LEFT JOIN cademycode_courses AS cd
                                         ON cs.current_career_path_id = cd.career_path_id
                        
                                         LEFT JOIN cademycode_student_jobs AS jd
                                         ON cs.job_id = jd.job_id; ''', conn
                                     )
logger.info('successfully querried cleaned data from the db')

[2025-09-14 12:16:44,061] main successfully querried cleaned data from the db


In [36]:
conn.close()
logger.info('CleanedData.db closes')

[2025-09-14 12:16:44,073] main CleanedData.db closes


In [37]:
Analytics_table_df.head(5)

,uuid,name,Date_of_Birth,sex,email,num_course_taken,career_path_name,hours_to_complete,job_category,avg_salary
0,1,Annabelle Avery,1943-07-03,F,annabelle_avery9376@woohoo.com,6.0,data scientist,20.0,HR,80000.0
1,2,Micah Rubio,1991-02-07,M,rubio6772@hmail.com,5.0,android developer,27.0,HR,80000.0
2,3,Hosea Dale,1989-12-07,M,hosea_dale8084@coldmail.com,8.0,android developer,27.0,HR,80000.0
3,4,Mariann Kirk,1988-07-31,F,kirk4005@hmail.com,7.0,machine learning engineer,35.0,education,61000.0
4,5,Lucio Alexander,1963-08-31,M,alexander9810@hmail.com,14.0,data analyst,12.0,HR,80000.0


In [38]:
Analytics_table_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7006 entries, 0 to 7005
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   uuid               7006 non-null   int64  
 1   name               7006 non-null   object 
 2   Date_of_Birth      7006 non-null   object 
 3   sex                7006 non-null   object 
 4   email              7006 non-null   object 
 5   num_course_taken   6653 non-null   float64
 6   career_path_name   6349 non-null   object 
 7   hours_to_complete  6349 non-null   float64
 8   job_category       7001 non-null   object 
 9   avg_salary         7001 non-null   float64
dtypes: float64(3), int64(1), object(6)
memory usage: 547.5+ KB


In [39]:
Analytics_table_df.to_csv('Customer_data.csv', index=False, sep =',', header=True)
logger.info('Data successfully written to a CSV_FILE')

[2025-09-14 12:16:44,139] main Data successfully written to a CSV_FILE


In [40]:
logger.info('We are at the end')

[2025-09-14 12:16:44,155] main We are at the end


# *AUTOMATION*